# A/B Testing & Experimentation Analysis

## 🧪 Business Context

A/B testing is the gold standard for causal inference in product development. Whether testing a new UI, a pricing change, or a recommendation algorithm, rigorous statistical analysis is required to ensure that observed differences are real and not due to chance.

## 📊 Objectives

1. Design the experiment (Sample Size & Power Analysis)
2. Analyze results using Frequentist methods (T-test, Z-test)
3. Analyze results using Bayesian methods (Beta-Binomial)
4. Check for Novelty Effects and Simpson's Paradox
5. Provide actionable recommendations based on statistical significance and practical significance

## 🔧 Methodology

- **Data**: Synthetic experiment data (Control vs Treatment)
- **Techniques**: Power Analysis, Z-test, T-test, Bayesian Inference, Bootstrap
- **Metrics**: Conversion Rate, Lift, P-value, Confidence Interval, Posterior Probability

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import TTestIndPower, GofChisquarePower
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

print('✓ Libraries loaded successfully')

## 1. Experiment Design & Power Analysis

Determining the required sample size before running the test.

In [ ]:
# Parameters
effect_size = 0.02 # Detect a 2% absolute difference (e.g., 10% vs 12%)
alpha = 0.05       # Significance level (5%)
power = 0.80       # Statistical power (80%)

analysis = TTestIndPower()
sample_size = analysis.solve_power(effect_size=0.1, power=power, alpha=alpha)
print(f"Required Sample Size per Group (Cohen's d=0.1): {int(sample_size)}")

# Power Curve
fig = plt.figure(figsize=(10, 6))
analysis.plot_power(dep_var='nobs', nobs=np.arange(50, 5000), effect_size=[0.05, 0.1, 0.2], alpha=0.05)
plt.title('Power Analysis: Sample Size vs Power')
plt.axhline(0.8, color='red', linestyle='--', label='Target Power (0.8)')
plt.legend()
plt.grid(True)
plt.savefig('outputs/power_analysis.png')
plt.show()

## 2. Data Generation

Simulating experiment results for Control and Treatment groups.

In [ ]:
def generate_ab_data(n=2000):
    np.random.seed(42)
    
    # Control: 10% conversion rate
    control_converted = np.random.binomial(1, 0.10, n)
    control_revenue = control_converted * np.random.lognormal(4, 0.5, n)
    
    # Treatment: 11.5% conversion rate (True lift)
    treatment_converted = np.random.binomial(1, 0.115, n)
    treatment_revenue = treatment_converted * np.random.lognormal(4, 0.5, n)
    
    df = pd.DataFrame({
        'Group': ['Control']*n + ['Treatment']*n,
        'Converted': np.concatenate([control_converted, treatment_converted]),
        'Revenue': np.concatenate([control_revenue, treatment_revenue])
    })
    
    return df

df = generate_ab_data(5000)
print(f"Dataset Shape: {df.shape}")
display(df.groupby('Group').mean())

## 3. Frequentist Analysis (Z-Test)

Testing for difference in proportions (Conversion Rate).

In [ ]:
control_results = df[df['Group'] == 'Control']['Converted']
treatment_results = df[df['Group'] == 'Treatment']['Converted']

n_con = control_results.count()
n_treat = treatment_results.count()
successes = [control_results.sum(), treatment_results.sum()]
nobs = [n_con, n_treat]

z_stat, p_val = proportions_ztest(successes, nobs)

(lower_con, lower_treat), (upper_con, upper_treat) = proportion_confint(successes, nobs=nobs, alpha=0.05)

print(f'Z-Statistic: {z_stat:.4f}')
print(f'P-Value: {p_val:.4f}')
print(f'Control CR: {successes[0]/n_con:.2%} [{lower_con:.2%}, {upper_con:.2%}]')
print(f'Treatment CR: {successes[1]/n_treat:.2%} [{lower_treat:.2%}, {upper_treat:.2%}]')

if p_val < 0.05:
    print("\nResult: Statistically Significant Difference!")
else:
    print("\nResult: No Significant Difference.")

## 4. Bayesian Analysis

Calculating the probability that Treatment is better than Control.

In [ ]:
# Beta Priors (Weak prior: alpha=1, beta=1)
alpha_prior = 1
beta_prior = 1

posterior_control = stats.beta(alpha_prior + successes[0], beta_prior + n_con - successes[0])
posterior_treatment = stats.beta(alpha_prior + successes[1], beta_prior + n_treat - successes[1])

# Sampling from Posteriors
samples = 10000
samples_control = posterior_control.rvs(samples)
samples_treatment = posterior_treatment.rvs(samples)

prob_treatment_better = (samples_treatment > samples_control).mean()

print(f"Probability Treatment > Control: {prob_treatment_better:.2%}")

# Visualization
plt.figure(figsize=(10, 6))
sns.kdeplot(samples_control, fill=True, label='Control Posterior')
sns.kdeplot(samples_treatment, fill=True, label='Treatment Posterior')
plt.title('Bayesian Posterior Distributions of Conversion Rate')
plt.xlabel('Conversion Rate')
plt.legend()
plt.savefig('outputs/bayesian_posterior.png')
plt.show()

## 5. Revenue Analysis (Bootstrap)

Analyzing Revenue Per User (RPU) which is non-normal.

In [ ]:
rev_control = df[df['Group'] == 'Control']['Revenue']
rev_treatment = df[df['Group'] == 'Treatment']['Revenue']

# Bootstrap Mean Difference
diffs = []
for _ in range(1000):
    c_sample = rev_control.sample(frac=1, replace=True)
    t_sample = rev_treatment.sample(frac=1, replace=True)
    diffs.append(t_sample.mean() - c_sample.mean())

ci_lower = np.percentile(diffs, 2.5)
ci_upper = np.percentile(diffs, 97.5)

plt.figure(figsize=(10, 6))
sns.histplot(diffs, kde=True)
plt.axvline(0, color='red', linestyle='--')
plt.axvline(ci_lower, color='green', linestyle=':')
plt.axvline(ci_upper, color='green', linestyle=':')
plt.title('Bootstrap Distribution of Difference in Revenue Per User')
plt.xlabel('Difference (Treatment - Control)')
plt.savefig('outputs/bootstrap_revenue.png')
plt.show()

print(f"Mean Difference: ${np.mean(diffs):.2f}")
print(f"95% CI: [${ci_lower:.2f}, ${ci_upper:.2f}]")

## 6. Conclusion

Final recommendation.

In [ ]:
print("="*60)
print("EXPERIMENT CONCLUSION")
print("="*60)
print(f"1. Conversion Rate: Treatment ({successes[1]/n_treat:.2%}) vs Control ({successes[0]/n_con:.2%}).")
print(f"   - Lift: {(successes[1]/n_treat - successes[0]/n_con)/(successes[0]/n_con):.2%}")
print(f"   - Significance: {p_val < 0.05} (p={p_val:.4f})")
print(f"2. Bayesian Probability: {prob_treatment_better:.1%} chance Treatment is better.")
print(f"3. Revenue Impact: Expected lift of ${np.mean(diffs):.2f} per user.")
print("4. Recommendation: ROLL OUT Treatment to 100% of users.")